# Window Detection and Permutation ANOVA

## For Best Frans

In [2]:
from analyses.response_window_analysis import run_permutation_anova_by_window
from analyses.spike_count import extract_spike_counts_from_windows
import pandas as pd
from analyses.response_window_finder.threshold_window_detection import compute_timebinned_spikecount_per_neuron, \
    z_score, threshold_and_fill_gap, extract_consecutive_ranges, remove_consecutive_tuples, \
    find_corresponding_values_for_index_ranges
from tqdm import tqdm
import numpy as np
from analyses.data_readers.recording_metadata_reader import RecordingMetadataReader

In [ ]:
# Detect windows for best frans
prelim = RecordingMetadataReader().get_metadata_for_preliminary_analysis()
results = []
bin_size = 0.05  # in sec
rounded_time = np.round(np.arange(bin_size, 3.50, bin_size), 2)
monkey_group = 'Best Frans'
for _, row in tqdm(prelim.iterrows(), total = len(prelim), desc = "Processing each recording day..."):
    round_no = row['Round No.']
    date = str(row['Date'].strftime('%Y-%m-%d'))
    timebin_spikecount_list = compute_timebinned_spikecount_per_neuron(date, round_no, bin_size, monkey_group)
    for _, r in timebin_spikecount_list.iterrows():
        data = r['TotalSpikeCountList']
        neuron = r['NeuronID']
        normalized_data = z_score(data)
        thresh = 0.5
        change_points = threshold_and_fill_gap(normalized_data, thresh)
        windows = extract_consecutive_ranges(change_points)
        filtered_windows = remove_consecutive_tuples(windows)
        time_windows = find_corresponding_values_for_index_ranges(filtered_windows, rounded_time)
        if len(time_windows) > 0:
            for start_time, end_time in time_windows:
                results.append({
                    'NeuronID': neuron,
                    'WindowStart_ms': int(start_time * 1000),
                    'WindowEnd_ms': int(end_time * 1000)
                })


In [ ]:
# Detected windows
bf_results_df = pd.DataFrame(results)
bf_results_df = bf_results_df.sort_values(by=['NeuronID'])
bf_results_df

In [ ]:
# Extract spike counts in the windows
bf_final_df = extract_spike_counts_from_windows(bf_results_df)

In [ ]:
# Run Perm ANOVA on BestFrans
bestfrans_df = bf_final_df[bf_final_df['MonkeyGroup'] == 'Best Frans']
bf_perm_results, bf_sig_results = run_permutation_anova_by_window(bestfrans_df,
                                category_col='MonkeyName',
                                neuron_col='NeuronID',
                                count_col='SpikeCount',
                                window_start_col='WindowStart_ms',
                                window_end_col='WindowEnd_ms',
                                n_permutations=1000,
                                alpha=0.05,
                                plot=False)

In [ ]:
bf_sig_results

In [ ]:
bf_sig_results[['Date', 'Round No.', 'Cell']] = bf_sig_results['NeuronID'].str.split('_', n=2, expand=True)
bf_sig_results['Time Window'] = list(zip(bf_sig_results['WindowStart_ms'], bf_sig_results['WindowEnd_ms']))
bf_sig_results.head() # --- save this and share with Ed !!

In [ ]:
bf_sig_results.to_excel('bestfrans_panova_passed_cells_after_window_detection_updated.xlsx', index=False)

In [ ]:
# Get spike counts anova passed windows
bf_spike_count_for_sig_windows_anova_passed = extract_spike_counts_from_windows(bf_sig_results)

In [ ]:
bf_spike_count = bf_spike_count_for_sig_windows_anova_passed.copy()
bf_spike_count.head()

In [ ]:
bf_spike_count[['Location','Date', 'Round No.', 'Cell']] = bf_spike_count['NeuronID'].str.split('_', n=3, expand=True)
bf_spike_count['Time Window'] = list(zip(bf_spike_count['WindowStart_ms'], bf_spike_count['WindowEnd_ms']))
bf_spike_count.head()

In [ ]:
group_cols = ['Date', 'Round No.', 'Cell', 'Time Window','MonkeyName']
bf_spike_count['Date'] = bf_spike_count['Date'].astype(str)
bf_grouped_df = bf_spike_count.groupby(group_cols)['SpikeCount'].apply(list).reset_index()
bf_final_spike_count_df = bf_grouped_df.pivot(
    index=['Date', 'Round No.', 'Cell', 'Time Window'],
    columns='MonkeyName',
    values='SpikeCount'
).reset_index()
print(bf_final_spike_count_df) # -- save this and share with Ed!!

In [ ]:
bf_final_spike_count_df.to_excel('bestfrans_spike_counts_for_all_panova_passed_time_windowed_updated.xlsx', index=False)

### For Zombies

In [ ]:
# Detect windows for zombies
prelim = RecordingMetadataReader().get_metadata_for_preliminary_analysis()
results = []
bin_size = 0.05  # in sec
rounded_time = np.round(np.arange(bin_size, 3.50, bin_size), 2)
monkey_group = 'Zombies'
for _, row in tqdm(prelim.iterrows(), total = len(prelim), desc = "Processing each recording day..."):
    round_no = row['Round No.']
    date = str(row['Date'].strftime('%Y-%m-%d'))
    timebin_spikecount_list = compute_timebinned_spikecount_per_neuron(date, round_no, bin_size, monkey_group)
    for _, r in timebin_spikecount_list.iterrows():
        data = r['TotalSpikeCountList']
        neuron = r['NeuronID']
        normalized_data = z_score(data)
        thresh = 0.5
        change_points = threshold_and_fill_gap(normalized_data, thresh)
        windows = extract_consecutive_ranges(change_points)
        filtered_windows = remove_consecutive_tuples(windows)
        time_windows = find_corresponding_values_for_index_ranges(filtered_windows, rounded_time)
        if len(time_windows) > 0:
            for start_time, end_time in time_windows:
                results.append({
                    'NeuronID': neuron,
                    'WindowStart_ms': int(start_time * 1000),
                    'WindowEnd_ms': int(end_time * 1000)
                })


In [ ]:
# Detected windows
zombies_results_df = pd.DataFrame(results)
zombies_results_df = zombies_results_df.sort_values(by=['NeuronID'])
zombies_results_df.head()

In [ ]:
# Extract spike counts in the windows
zombies_extracted_spikes_df = extract_spike_counts_from_windows(zombies_results_df)

In [ ]:
# Run Perm ANOVA on BestFrans
zombies_df = zombies_extracted_spikes_df[zombies_extracted_spikes_df['MonkeyGroup'] == monkey_group]
_, zom_sig_results = run_permutation_anova_by_window(zombies_df,
                                category_col='MonkeyName',
                                neuron_col='NeuronID',
                                count_col='SpikeCount',
                                window_start_col='WindowStart_ms',
                                window_end_col='WindowEnd_ms',
                                n_permutations=1000,
                                alpha=0.05,
                                plot=False)

In [ ]:
zom_sig_results.head()

In [ ]:
zom_sig_copy= zom_sig_results.copy()
zom_sig_copy[['Date', 'Round No.', 'Cell']] = zom_sig_results['NeuronID'].str.split('_', n=2, expand=True)
zom_sig_copy['Time Window'] = list(zip(zom_sig_copy['WindowStart_ms'], zom_sig_copy['WindowEnd_ms']))
zom_sig_copy.head() # --- save this and share with Ed !!

In [ ]:
zom_sig_copy.to_excel('zombies_panova_passed_cells_after_window_detection_updated.xlsx', index=False)
print('Done!')

In [ ]:
# Get spike counts anova passed windows
zom_spike_count_for_sig_windows_anova_passed = extract_spike_counts_from_windows(zom_sig_copy)

In [ ]:
zom_spike_count = zom_spike_count_for_sig_windows_anova_passed.copy()
zom_spike_count.head()

In [ ]:
zom_spike_count[['Location','Date', 'Round No.', 'Cell']] = zom_spike_count['NeuronID'].str.split('_', n=3, expand=True)
zom_spike_count['Time Window'] = list(zip(zom_spike_count['WindowStart_ms'], zom_spike_count['WindowEnd_ms']))
zom_spike_count.head()

In [ ]:
zom_spike_count.head()
group_cols = ['Date', 'Round No.', 'Cell', 'Time Window', 'MonkeyName']
zom_spike_count['Date'] = zom_spike_count['Date'].astype(str)
zom_grouped_df = zom_spike_count.groupby(group_cols)['SpikeCount'].apply(list).reset_index()
zom_final_spike_count_df = zom_grouped_df.pivot(
    index=['Date', 'Round No.', 'Cell', 'Time Window'],
    columns='MonkeyName',
    values='SpikeCount'
).reset_index()
print(zom_final_spike_count_df.head())  # -- save this and share with Ed!!

In [ ]:
zom_final_spike_count_df.to_excel('zombies_spike_counts_for_all_panova_passed_time_windowed_updated.xlsx', index=False)

# For Instigators

In [12]:
import pandas as pd
donkey = pd.read_pickle('/home/connorlab/Documents/JulieData/Cortana/analysis_cache/si_sorted_Instigators_significant_neurons_pANOVA_passed.pkl')
donkey.to_excel('instigators_panova_passed_neurons.xlsx', index=False)

In [4]:
# Detected windows
ig_results_df = pd.DataFrame(results)
ig_results_df = ig_results_df.sort_values(by=['NeuronID'])
ig_results_df

,NeuronID,WindowStart_ms,WindowEnd_ms
0,AMG_2023-09-26_1_Channel.C_014_Unit 1,150,650
1,AMG_2023-09-26_1_Channel.C_022_Unit 1,50,600
2,AMG_2023-09-26_1_Channel.C_025_Unit 1,400,500
3,AMG_2023-09-26_1_Channel.C_025_Unit 1,600,750
4,AMG_2023-09-26_1_Channel.C_025_Unit 1,850,950
...,...,...,...
431,Unknown_2023-11-08_3_Channel.C_024,1500,1600
432,Unknown_2023-11-08_3_Channel.C_028,150,250
433,Unknown_2023-11-08_3_Channel.C_028,700,900
435,Unknown_2023-11-08_3_Channel.C_030,1300,1450


In [7]:
from data_access.spike_source import SISortedSpikeSource
from data_access.spike_source import SpikeSource

# Extract spike counts in the windows
# ig_final_df = extract_spike_counts_from_windows(ig_results_df)
source: SpikeSource = SISortedSpikeSource(cache_subdir="sorted_spike_cache_filtered", pre_filtered=True)
ig_windows = pd.read_pickle('/home/connorlab/Documents/JulieData/Cortana/analysis_cache/si_sorted_Instigators_significant_windows_pANOVA_passed.pkl')
ig_final_df = extract_spike_counts_from_windows(ig_windows, source)

Extracting spike counts: 100%|██████████| 50/50 [00:00<00:00, 59.40it/s]


In [8]:
ig_final_df

,NeuronID,MonkeyName,MonkeyGroup,TaskField,WindowStart_ms,WindowEnd_ms,SpikeCount
0,AMG_2023-09-26_2_Channel.C_000_Unit 1,37I,Stranger Things,1695750846219000,1250,1400,0
1,AMG_2023-09-26_2_Channel.C_000_Unit 1,0FL,Stranger Things,1695750846372000,1250,1400,0
2,AMG_2023-09-26_2_Channel.C_000_Unit 1,68F,Best Frans,1695750846439000,1250,1400,0
3,AMG_2023-09-26_2_Channel.C_000_Unit 1,58I,Stranger Things,1695750846583000,1250,1400,0
4,AMG_2023-09-26_2_Channel.C_000_Unit 1,58I,Stranger Things,1695750846672000,1250,1400,0
...,...,...,...,...,...,...,...
17950,Unknown_2023-11-08_2_Channel.C_019_Unit 2,101G,Best Frans,1699479823929000,200,400,2
17951,Unknown_2023-11-08_2_Channel.C_019_Unit 2,26J,Stranger Things,1699479826403000,200,400,2
17952,Unknown_2023-11-08_2_Channel.C_019_Unit 2,G701,Best Frans,1699479827365000,200,400,2
17953,Unknown_2023-11-08_2_Channel.C_019_Unit 2,36J,Stranger Things,1699479813031000,200,400,3


In [13]:
ig_final_df[['Date', 'Round No.', 'Cell']] = ig_final_df['NeuronID'].str.split('_', n=2, expand=True)
ig_final_df['Time Window'] = list(zip(ig_final_df['WindowStart_ms'], ig_final_df['WindowEnd_ms']))

In [14]:
ig_spike_count[['Location','Date', 'Round No.', 'Cell']] = ig_spike_count['NeuronID'].str.split('_', n=3, expand=True)
ig_spike_count['Time Window'] = list(zip(ig_spike_count['WindowStart_ms'], ig_spike_count['WindowEnd_ms']))
ig_spike_count.head()

,NeuronID,MonkeyName,MonkeyGroup,TaskField,WindowStart_ms,WindowEnd_ms,SpikeCount,Date,Round No.,Cell,Time Window,Location
0,AMG_2023-09-26_2_Channel.C_000_Unit 1,37I,Stranger Things,1695750846219000,1250,1400,0,2023-09-26,2,Channel.C_000_Unit 1,"(1250, 1400)",AMG
1,AMG_2023-09-26_2_Channel.C_000_Unit 1,0FL,Stranger Things,1695750846372000,1250,1400,0,2023-09-26,2,Channel.C_000_Unit 1,"(1250, 1400)",AMG
2,AMG_2023-09-26_2_Channel.C_000_Unit 1,68F,Best Frans,1695750846439000,1250,1400,0,2023-09-26,2,Channel.C_000_Unit 1,"(1250, 1400)",AMG
3,AMG_2023-09-26_2_Channel.C_000_Unit 1,58I,Stranger Things,1695750846583000,1250,1400,0,2023-09-26,2,Channel.C_000_Unit 1,"(1250, 1400)",AMG
4,AMG_2023-09-26_2_Channel.C_000_Unit 1,58I,Stranger Things,1695750846672000,1250,1400,0,2023-09-26,2,Channel.C_000_Unit 1,"(1250, 1400)",AMG


In [15]:
group_cols = ['Date', 'Round No.', 'Cell', 'Time Window','MonkeyName']
ig_spike_count['Date'] = ig_spike_count['Date'].astype(str)
ig_grouped_df = ig_spike_count.groupby(group_cols)['SpikeCount'].apply(list).reset_index()
ig_final_spike_count_df = ig_grouped_df.pivot(
    index=['Date', 'Round No.', 'Cell', 'Time Window'],
    columns='MonkeyName',
    values='SpikeCount'
).reset_index()
print(ig_final_spike_count_df) # -- save this and share with Ed!!

MonkeyName        Date Round No.                  Cell   Time Window  \
0           2023-09-26         2  Channel.C_000_Unit 1  (1250, 1400)   
1           2023-09-26         2  Channel.C_010_Unit 1    (200, 500)   
2           2023-09-26         2  Channel.C_015_Unit 1    (250, 500)   
3           2023-09-26         2  Channel.C_017_Unit 1    (150, 400)   
4           2023-09-26         2  Channel.C_022_Unit 1    (150, 250)   
5           2023-09-26         3  Channel.C_005_Unit 3   (900, 1000)   
6           2023-09-29         3  Channel.C_016_Unit 1    (150, 450)   
7           2023-09-29         3  Channel.C_019_Unit 1    (800, 950)   
8           2023-10-03         3  Channel.C_002_Unit 2    (650, 750)   
9           2023-10-03         4  Channel.C_010_Unit 1    (100, 400)   
10          2023-10-04         1  Channel.C_008_Unit 1    (150, 400)   
11          2023-10-04         1  Channel.C_008_Unit 1  (1050, 1200)   
12          2023-10-04         1  Channel.C_014_Unit 1    (150, 

In [16]:
ig_final_spike_count_df.head()

MonkeyName,Date,Round No.,Cell,Time Window,09X,0EX,0FL,101G,110E,114J,...,70G,7124,72X,79G,86I,87J,94B,DF2I,G701,G942
0,2023-09-26,2,Channel.C_000_Unit 1,"(1250, 1400)","[0, 1, 0, 2, 1, 2, 0, 2]","[0, 0, 1, 3, 1, 0, 0, 0, 0, 2]","[0, 0, 0, 0, 2, 0, 1, 2, 2]","[0, 0, 0, 0, 0, 0, 1, 2, 2, 1]","[0, 0, 0, 0, 0, 0, 2, 0, 2, 4]","[0, 0, 2, 0, 0, 2, 0, 0, 4, 0]",...,"[0, 0, 0, 0, 2, 1, 1, 0, 1, 2]","[1, 0, 0, 2, 0, 1, 2, 1, 2]","[0, 0, 0, 0, 0, 2, 1, 2, 1, 3]","[1, 2, 0, 2, 2, 2, 1, 1, 2, 3]","[1, 0, 0, 2, 1, 2, 2, 0, 4, 2]","[1, 0, 2, 0, 0, 0, 2, 0, 2, 0]","[0, 2, 0, 0, 2, 0, 2, 2, 0]","[0, 0, 2, 1, 0, 1, 0, 2, 2, 2]","[0, 0, 0, 0, 4, 1, 2, 2, 2, 4]","[1, 1, 1, 2, 1, 2, 0, 0, 1, 4]"
1,2023-09-26,2,Channel.C_010_Unit 1,"(200, 500)","[6, 6, 13, 14, 10, 9, 5, 5]","[6, 1, 5, 11, 9, 10, 2, 2, 7, 10]","[5, 4, 5, 1, 2, 5, 3, 4, 6]","[4, 9, 2, 11, 8, 4, 2, 8, 7, 10]","[5, 2, 1, 4, 10, 7, 3, 9, 7, 13]","[6, 1, 7, 8, 4, 5, 8, 9, 8, 11]",...,"[0, 1, 7, 3, 3, 5, 10, 5, 12, 9]","[8, 8, 3, 5, 14, 9, 11, 8, 4]","[4, 6, 2, 7, 7, 10, 9, 9, 10, 14]","[4, 3, 3, 4, 7, 5, 6, 10, 10, 7]","[3, 10, 10, 2, 11, 7, 5, 9, 9, 6]","[2, 11, 3, 6, 3, 5, 4, 4, 10, 6]","[0, 4, 4, 11, 7, 6, 3, 8, 5]","[9, 4, 9, 6, 12, 6, 12, 8, 13, 5]","[8, 10, 6, 9, 5, 12, 11, 16, 8, 15]","[9, 14, 11, 10, 11, 7, 12, 10, 16, 10]"
2,2023-09-26,2,Channel.C_015_Unit 1,"(250, 500)","[0, 0, 0, 0, 0, 0, 2, 3]","[0, 0, 0, 0, 0, 1, 0, 0, 2, 0]","[0, 0, 0, 0, 1, 0, 0, 0, 1]","[0, 0, 0, 1, 0, 0, 2, 1, 0, 1]","[0, 1, 1, 1, 1, 0, 0, 0, 1, 0]","[0, 1, 0, 0, 0, 1, 1, 0, 0, 0]",...,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 1, 0, 0]","[1, 2, 0, 0, 1, 0, 0, 1, 0, 0]","[0, 0, 0, 0, 3, 0, 1, 0, 1, 1]","[0, 1, 0, 1, 0, 0, 1, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0]","[0, 0, 0, 0, 0, 1, 0, 0, 1]","[1, 0, 0, 1, 0, 2, 0, 1, 0, 0]","[0, 2, 0, 1, 0, 1, 0, 1, 0, 0]","[0, 0, 0, 1, 1, 0, 0, 1, 0, 0]"
3,2023-09-26,2,Channel.C_017_Unit 1,"(150, 400)","[0, 0, 1, 1, 1, 2, 3, 1]","[1, 0, 1, 0, 1, 0, 0, 1, 1, 2]","[2, 0, 0, 1, 1, 3, 0, 2, 2]","[1, 0, 0, 1, 0, 0, 0, 2, 1, 0]","[0, 0, 0, 2, 0, 0, 1, 0, 0, 0]","[0, 0, 1, 0, 2, 1, 1, 0, 0, 0]",...,"[0, 1, 2, 1, 1, 0, 1, 1, 0, 0]","[0, 0, 0, 1, 0, 0, 2, 0, 3]","[1, 1, 0, 0, 0, 0, 1, 1, 0, 0]","[0, 2, 0, 1, 0, 0, 0, 1, 1, 2]","[0, 0, 2, 0, 1, 1, 0, 0, 1, 2]","[0, 0, 1, 0, 0, 1, 1, 1, 2, 2]","[1, 1, 0, 1, 1, 0, 0, 1, 0]","[3, 1, 0, 0, 0, 1, 1, 0, 2, 1]","[1, 0, 0, 1, 0, 0, 2, 1, 1, 1]","[1, 1, 1, 1, 0, 0, 0, 1, 0, 0]"
4,2023-09-26,2,Channel.C_022_Unit 1,"(150, 250)","[4, 2, 5, 4, 3, 3, 5, 3]","[2, 4, 10, 0, 1, 3, 4, 4, 3, 4]","[2, 2, 3, 3, 4, 6, 5, 5, 3]","[1, 1, 4, 1, 0, 1, 3, 5, 3, 1]","[5, 9, 6, 2, 1, 4, 3, 4, 7, 3]","[2, 2, 1, 2, 0, 6, 7, 2, 2, 3]",...,"[1, 7, 6, 4, 0, 7, 3, 3, 6, 10]","[3, 5, 3, 4, 6, 5, 4, 5, 8]","[2, 4, 6, 3, 4, 2, 2, 1, 2, 4]","[0, 2, 2, 6, 1, 6, 5, 6, 10, 7]","[3, 3, 1, 5, 3, 3, 2, 1, 2, 3]","[8, 4, 8, 10, 4, 6, 3, 4, 5, 6]","[3, 3, 7, 5, 3, 9, 3, 3, 3]","[1, 1, 5, 3, 8, 5, 3, 3, 5, 4]","[3, 2, 1, 3, 3, 6, 5, 6, 4, 1]","[1, 3, 3, 4, 3, 5, 2, 3, 3, 5]"


In [17]:
ig_final_spike_count_df.to_excel('instigators_spike_counts_for_all_panova_passed_time_windowed_cells_20260526.xlsx', index=False)

In [29]:
import pandas as pd
see = pd.read_pickle('/home/connorlab/Documents/JulieData/Cortana/exploded_spike_cache/2023-10-05_round_2.pkl')
see['NeuronID'].nunique()

17

,TaskField,MonkeyId,MonkeyGroup,MonkeyName,Channel,SpikeTimes,EpochStartStop,BaseChannel,Date,Round No.,Location,NeuronID
0,1696533506269000,6498,Stranger Things,37I,Channel.C_007_Unit 1,"[14.49435, 14.50885, 14.51735, 14.70485, 14.72...","(14.2333, 16.875)",Channel.C_007,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_007_Unit 1
1,1696533506269000,6498,Stranger Things,37I,Channel.C_007_Unit 2,[],"(14.2333, 16.875)",Channel.C_007,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_007_Unit 2
2,1696533506269000,6498,Stranger Things,37I,Channel.C_009_Unit 1,[14.49375],"(14.2333, 16.875)",Channel.C_009,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_009_Unit 1
3,1696533506269000,6498,Stranger Things,37I,Channel.C_009_Unit 2,"[14.35405, 14.5535, 14.8026, 14.8446, 16.0336,...","(14.2333, 16.875)",Channel.C_009,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_009_Unit 2
4,1696533506269000,6498,Stranger Things,37I,Channel.C_010_Unit 1,[],"(14.2333, 16.875)",Channel.C_010,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_010_Unit 1
...,...,...,...,...,...,...,...,...,...,...,...,...
10348,1696533528670000,3460,Instigators,48Z,Channel.C_013,[2428.68365],"(2426.47105, 2428.8128)",Channel.C_013,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_013
10351,1696533528670000,3460,Instigators,48Z,Channel.C_020,"[2426.4869, 2426.97185, 2426.99535, 2428.4187,...","(2426.47105, 2428.8128)",Channel.C_020,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_020
10353,1696533528670000,3460,Instigators,48Z,Channel.C_022,"[2426.5312, 2426.80715, 2426.86135, 2426.95435...","(2426.47105, 2428.8128)",Channel.C_022,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_022
10356,1696533528670000,3460,Instigators,48Z,Channel.C_026,[2426.59505],"(2426.47105, 2428.8128)",Channel.C_026,2023-10-05,2,AMG,AMG_2023-10-05_2_Channel.C_026


In [25]:
see['NeuronID'].nunique()

34

In [32]:
import os

folder = "/home/connorlab/Documents/JulieData/Cortana/sorted_spike_cache_filtered"
total_neurons = 0
for file in os.listdir(folder):

    if not file.endswith(".pkl"):
        continue

    path = os.path.join(folder, file)

    df = pd.read_pickle(path)
    num_neuron = df['NeuronID'].nunique()
    total_neurons += num_neuron

print(total_neurons)

331
